# 05 - Feed-forward on the classifier's decision

This is the paper's main result as a runnable experiment (paper Sec. IX). Per shot:

1. A recorded readout trace plays out of the DAC and comes back through the loopback.
2. The tProc fires the **classify trigger** with the readout. NN_live classifies a 576-sample window of the low-latency
   boxcar stream and produces its decision 9 cycles (16.3 ns) after the last sample.
3. The decision crosses into the DAC and tProc clock domains. The tProc has already scheduled a **conditional pulse** on
   generator 0, together with a **window marker** for the gate (`dac_gate`).
4. The gate **passes** the pulse if NN_live decided 1 for this shot and **blanks** it otherwise. A late decision, or one
   for the wrong shot, also blanks it.
5. A second capture records whether a pulse came back.

The notebook measures, shot by shot: whether a pulse came back, the gate's record, NN_live's own decision, and the loop
latency from the window's last sample to the pulse's 50% point at the readout buffer.

**You need:** a board running our v1.43 firmware with its server (`host/start_nameserver.ipynb`), the loopback cable, the
board's `k`/`c` from notebook 01, and test traces (`data/test20k_real_X.npy` / `_y.npy`, notebook 02 explains them).
The gate is left disabled at the end.

In [ ]:
from pathlib import Path

BOARD_HOST = "10.1.9.26"
NS_PORT, PROXY_NAME = 8888, "myqick"
DATA_DIR = Path("..")
CAL_DIR = Path("outputs/calibration")              # notebook 01's output
TRACES_X, TRACES_Y = "data/test20k_real_X.npy", "data/test20k_real_y.npy"
SCALE_REF = "data/test20k_real_X.npy"                   # one DAC scale for the whole set (as in the paper's runs)
CAL_ROWS = range(1000, 1008)                       # gate calibration shots (never scored)
FIRST_ROW, N_SHOTS = 2000, 50                      # shots to run in the loop
OUT_DIR = Path("outputs/feed_forward")

# operating point of the paper's loop (v1.43, boxcar path)
WINDOW = 576           # NN_live window, samples at 552.96 MS/s
NN_ADV = 0.1135        # us: the boxcar path delivers the same samples this much earlier than QICK's FIR path
MARGIN = 0.035         # us: decision time t_dec is this long after the window's programmed end
SLACK = -0.064         # us: conditional pulse time relative to t_dec (negative = before it; the gate waits for the decision)
PRE = 0.15             # us: the pulse-capture readout opens this long before the pulse
CP_GAIN, CP_LEN = 0.6, 0.2       # conditional pulse amplitude (fraction of full scale) and length (us)
TRIG_WIDTH, WIN_EXTRA = 0.5, 0.05
DELAY_LEAD = 4         # the gate window opens this many DAC fabric cycles before the pulse reaches it
BOXCAR = dict(sel=1, shift=7, m=11)   # NN_live on the boxcar path: 8-sample average, 11-sample moving sum, shift 7

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import Pyro4
from qick import QickConfig
from qick.asm_v2 import AveragerProgramV2

Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.PICKLE_PROTOCOL_VERSION = 4

soc = Pyro4.Proxy(Pyro4.locateNS(host=BOARD_HOST, port=NS_PORT).lookup(PROXY_NAME))
soccfg = QickConfig(soc.get_cfg())
print([l.strip() for l in str(soccfg).splitlines() if "built" in l][0])

FS_RO = soccfg["readouts"][0]["fs"] / 8            # decimated readout rate, MHz (552.96)
TICK = 1.0 / soccfg["tprocs"][0]["f_time"]         # tProc time unit, us
LIVE_LEN = WINDOW / FS_RO                          # us: NN_live's window
pins = soccfg["tprocs"][0]["output_pins"]
GATE_PIN = [i for i, p in enumerate(pins) if p[0] == "trig" and p[1] == 10][0]   # the gate's window marker (tProc trig_10)
pins.append(("trig", 11, 0, "NN classify trigger (trig_11_o)"))                   # NN_live's classify trigger
CLASSIFY_PIN = len(pins) - 1
print(f"gate marker pin {GATE_PIN}, classify pin {CLASSIFY_PIN}; window {WINDOW} samples = {1e3 * LIVE_LEN:.1f} ns")

## The board's custom blocks

The server exposes our blocks through a few calls:
- **`dac_gate`:** `gate_write(0, word)` configures it; `gate_stats()` reads its 21 status words.
- **NN_live:** `live_read_block(addr, n)` reads its registers and its per-shot record ring (512 records of 8 words).
- **Boxcar path:** `box_config(...)` sets it up.

Gate config word 0, from bit 0 up:
- bit 0: enable;
- bit 1: pass when the decision is 1;
- bit 2: pass when the decision is 0;
- bit 3: pass when the decision is stale;
- bits 15:8: `DELAY`, the window start in DAC fabric cycles after the marker;
- bits 31:16: the arrival threshold the gate uses to time marker-to-pulse.

Status word 10 holds the last window's flags: bit 0 passed, bit 1 stale, bit 2 the decision bit the gate saw.

In [ ]:
GAT4 = 0x47415434            # dac_gate's magic word in v1.41 and later
SHOT_MASK = (1 << 29) - 1    # NN_live's shot counter is 29 bits


def gate_cfg(en, pass1=1, pass0=0, pass_stale=0, delay=0, nz_thr=2000):
    return (int(en) | int(pass1) << 1 | int(pass0) << 2 | int(pass_stale) << 3
            | (int(delay) & 0xFF) << 8 | (int(nz_thr) & 0xFFFF) << 16)


def live_last_record():
    # NN_live's record for the newest classified window: word 0 = shot number, word 7 bit 0 = decision.
    n = int(soc.live_read_block(0, 3)[0])
    return np.asarray(soc.live_read_block(1024 + ((n - 1) % 512) * 8, 8), dtype=np.int64)


print("boxcar path:", soc.box_config(**BOXCAR))
magic = int(soc.gate_stats()[11])
assert magic == GAT4, f"gate magic {magic:08x}: this notebook needs the v1.41+ gate"

## Traces

The same replay as notebook 02: predistort with $(k, c)$, interpolate to 11,000 envelope samples, one DAC scale for the
whole set. Two differences from notebook 02:
- Only the part of the trace inside the 576-sample window is played. The rest would never be classified, and on a qubit
  the readout pulse stops with the window.
- Values are clipped to the generator's range.

In [ ]:
k = complex(np.load(CAL_DIR / f"k_{BOARD_HOST}.npy")); c = complex(np.load(CAL_DIR / f"c_{BOARD_HOST}.npy"))
Xall = np.load(DATA_DIR / TRACES_X, mmap_mode="r"); yall = np.load(DATA_DIR / TRACES_Y).astype(int)
Xref = np.load(DATA_DIR / SCALE_REF, mmap_mode="r")
zr = (np.asarray(Xref[:, 0::2]) + 1j * np.asarray(Xref[:, 1::2]) - c) / k
SCALE = 32766 * 0.99 / (max(np.abs(zr.real).max(), np.abs(zr.imag).max()) * 1.0001)
del zr
MAXV = soccfg["gens"][0]["maxv"]
xo, xn = np.linspace(0, 1, 770), np.linspace(0, 1, 11000)


def waveform(row):
    z = (np.asarray(Xall[row, 0::2]) + 1j * np.asarray(Xall[row, 1::2]) - c) / k
    w = np.empty(22000); w[0::2] = np.interp(xn, xo, z.real) * SCALE; w[1::2] = np.interp(xn, xo, z.imag) * SCALE
    m = len(w) // 2
    keep = min(m, int(np.ceil(m * WINDOW / 770 / 16)) * 16)       # play only the classified window
    return np.clip(w[:2 * keep], -MAXV, MAXV)


print(f"k = {k:.5f}, c = {c:.3f}, DAC scale {SCALE:.4f}, played samples per trace {len(waveform(0)) // 2}")

## The program

Times are in µs from the start of the shot:
- the trace starts at 0, and the readout trigger fires at `adc_trig_offset`;
- the classify trigger fires `NN_ADV` earlier, because the boxcar path delivers the same samples that much sooner;
- $t_\text{dec}$ = classify trigger + window + `MARGIN`;
- the conditional pulse and the gate marker are both scheduled at $t_\text{dec}$ + `SLACK`, unconditionally. The gate,
  not the tProc, decides whether the pulse reaches the DAC;
- a second readout opens `PRE` before the pulse and records what comes back;
- after the pulse, the tProc reads NN_live's result from input port 3 for bookkeeping only.

In [ ]:
BASE = {"gen_ch": 0, "ro_ch": 0, "nqz": 1, "reps": 1, "final_delay": 100, "readout_length": LIVE_LEN,
        "adc_trig_offset": 0.31, "freq": 50, "mode": "oneshot", "stdysel": "zero", "outsel": "product",
        "gain": 1, "phase": 0.0}
LIVE_PORT = 3


class FeedForwardShot(AveragerProgramV2):
    def __init__(self, soccfg, cfg):
        super().__init__(soccfg, reps=cfg["reps"], final_delay=cfg["final_delay"], cfg=cfg)

    def _initialize(self, cf):
        self.declare_gen(ch=cf["gen_ch"], nqz=cf["nqz"])
        self.declare_readout(ch=cf["ro_ch"], length=cf["readout_length"])
        self.add_envelope(ch=cf["gen_ch"], name="e", idata=cf["waveform_I"], qdata=cf["waveform_Q"])
        self.add_pulse(ch=cf["gen_ch"], name="p", style="arb", envelope="e", gain=cf["gain"], freq=cf["freq"],
                       phase=cf["phase"], outsel=cf["outsel"], mode=cf["mode"], stdysel=cf["stdysel"], ro_ch=cf["ro_ch"])
        self.add_pulse(ch=cf["gen_ch"], name="c1", style="const", length=cf["cp_len"], freq=cf["freq"], phase=0.0,
                       gain=cf["cp_gain"], ro_ch=cf["ro_ch"])
        self.add_readoutconfig(ch=cf["ro_ch"], name="r", freq=cf["freq"], gen_ch=cf["gen_ch"])

    def _body(self, cf):
        aoff = cf["adc_trig_offset"]
        self.pulse(ch=cf["gen_ch"], name="p")
        self.send_readoutconfig(ch=cf["ro_ch"], name="r", t=0)
        self.trigger(ros=[cf["ro_ch"]], t=aoff, width=cf["trig_width"])
        self.trigger(pins=[0, CLASSIFY_PIN], t=aoff - cf["nn_adv"], width=10)
        t_dec = aoff - cf["nn_adv"] + LIVE_LEN + cf["margin"]
        t_cp = t_dec + cf["slack"]
        self.trigger(ros=[cf["ro_ch"]], t=t_dec + aoff - cf["pre"], width=cf["trig_width"])
        self.pulse(ch=cf["gen_ch"], name="c1", t=t_cp)
        self.trigger(pins=[GATE_PIN], t=t_cp, width=cf["cp_len"] + cf["win_extra"])
        self.wait(t=t_cp + cf["cp_len"] + 0.1)
        self.asm_inst({"CMD": "DPORT_RD", "DST": str(LIVE_PORT)})
        self.write_dmem(0, "s_port_l")
        self.write_dmem(1, "s_port_h")


SETTINGS = dict(cp_gain=CP_GAIN, cp_len=CP_LEN, trig_width=TRIG_WIDTH, win_extra=WIN_EXTRA, nn_adv=NN_ADV,
                margin=MARGIN, pre=PRE)
N_AVG = max(4, int(0.5 * CP_LEN * FS_RO))          # samples averaged for the pulse plateau


def onset(cap):
    # 50% crossing of |IQ| relative to the pulse plateau, linearly interpolated, in samples; and the plateau.
    mag = np.hypot(cap[:, 0], cap[:, 1])
    plat = np.convolve(mag, np.ones(N_AVG) / N_AVG, mode="valid").max()
    i = int(np.argmax(mag >= 0.5 * plat))
    if i == 0:
        return np.nan, plat
    return i - 1 + (0.5 * plat - mag[i - 1]) / (mag[i] - mag[i - 1]), plat


def run_shot(row, slack):
    w = waveform(row)
    cfg = dict(BASE, waveform_I=w[0::2], waveform_Q=w[1::2], slack=slack, **SETTINGS)
    iq = np.asarray(FeedForwardShot(soccfg, cfg).acquire_decimated(soc, progress=False))
    cap = iq.reshape(-1, iq.shape[-2], 2)[-1]                       # the second capture: the pulse
    port = np.asarray(soc.tproc.read_mem("dmem", length=2)).reshape(-1).astype(np.int64)
    rec = live_last_record()
    gate = np.array(soc.gate_stats(), dtype=np.int64)
    on, plat = onset(cap)
    # loop latency at the buffer: pulse 50% point minus the window's last sample, both in readout 0's time base
    t1 = round(BASE["adc_trig_offset"] / TICK) * TICK
    t_dec = BASE["adc_trig_offset"] - NN_ADV + LIVE_LEN + MARGIN
    t2 = round((t_dec + BASE["adc_trig_offset"] - PRE) / TICK) * TICK
    L_buf = 1e3 * (t2 + on / FS_RO - t1 - (WINDOW - 1) / FS_RO)
    return dict(cap=cap, plat=plat, gate=gate, dec=int(rec[7] & 1), shot=int(rec[0]),
                fresh=(int(np.uint32(port[1])) & SHOT_MASK) == rec[0], L_buf=L_buf)

## Calibrate the gate

With the gate set to pass everything, a few shots measure how many DAC fabric cycles the pulse takes from the marker to
the gate's input (status word 7). The window then opens `DELAY_LEAD` cycles before it arrives. The same shots give the
full pulse's plateau, the reference for "a pulse came back".

In [ ]:
soc.gate_write(0, gate_cfg(en=1, pass1=1, pass0=1, pass_stale=1))
cal = [run_shot(r, 0.0) for r in CAL_ROWS]
marker_to_pulse = [int(r["gate"][7]) for r in cal]
DELAY = min(marker_to_pulse) - DELAY_LEAD
PLAT_PASS = float(np.median([r["plat"] for r in cal]))
flags = int(soc.gate_stats()[10])
print(f"marker -> pulse {sorted(set(marker_to_pulse))} cycles -> DELAY {DELAY}; full-pulse plateau {PLAT_PASS:.0f}")
print(f"gate safety options: EARLY_BLANK {flags >> 8 & 1}, SHOT_BIND {flags >> 9 & 1}, FAIL_SAFE {flags >> 10 & 1}")

## Run the loop

Gate configuration: pass on a decision of 1, blank on 0, blank when stale.

In [ ]:
soc.gate_write(0, gate_cfg(en=1, pass1=1, pass0=0, pass_stale=0, delay=DELAY))
late0, rej0 = int(soc.gate_stats()[12]), int(soc.gate_stats()[19])
rows = np.arange(FIRST_ROW, FIRST_ROW + N_SHOTS)
res, t0 = [], time.time()
try:
    for i, r in enumerate(rows):
        res.append(run_shot(r, SLACK))
        if i % 10 == 0:
            print(f"  {i + 1}/{len(rows)}")
finally:
    soc.gate_write(0, gate_cfg(en=0, delay=DELAY))
    print(f"done in {time.time() - t0:.0f} s; gate left disabled")
late = int(soc.gate_stats()[12]) - late0
rejects = int(soc.gate_stats()[19]) - rej0

## Score

- **In-loop fidelity:** "a pulse came back" against the trace's label. This is the classification as the loop delivers
  it, with every loop failure counted as an error.
- **Correct:** the pulse, the gate and NN_live's decision all agree on a fresh decision.
- **Unsafe:** a pulse came back without a fresh decision of 1 for its own shot. This must be 0.

In [ ]:
y = yall[rows]
g = np.array([r["gate"] for r in res])
passed, stale, gbit = (g[:, 10] & 1) > 0, (g[:, 10] & 2) > 0, (g[:, 10] >> 2) & 1
came = np.array([r["plat"] for r in res]) > 0.5 * PLAT_PASS
dec = np.array([r["dec"] for r in res]); fresh = np.array([r["fresh"] for r in res])
correct = (came == passed) & ~stale & (gbit == dec)
unsafe = came & ~(~stale & (gbit == dec) & (dec == 1))


def fidelity(pred, y):
    p10, p01 = np.mean(pred[y == 0] != 0), np.mean(pred[y == 1] != 1)
    return 100 * (1 - (p10 + p01) / 2), 100 * p10, 100 * p01


print(f"in-loop fidelity {fidelity(came.astype(int), y)[0]:.1f}%   NN_live decisions alone {fidelity(dec, y)[0]:.1f}%   (n = {len(y)})")
print(f"pulse == gate == decision: {correct.sum()}/{len(y)};  stale {stale.sum()}, late {late}, unsafe {unsafe.sum()}, "
      f"binding rejects {rejects}, fresh {fresh.sum()}")
L = np.array([r["L_buf"] for r in res])[came]
print(f"loop latency at the buffer (shots with a pulse): median {np.nanmedian(L):.1f} ns, "
      f"range {np.nanmin(L):.1f}-{np.nanmax(L):.1f} ns")

t = np.arange(res[0]["cap"].shape[0]) / FS_RO * 1e3
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for want, col, lab in ((1, "C3", "decision 1 (passed)"), (0, "C0", "decision 0 (blanked)")):
    i = np.flatnonzero(dec == want)
    if len(i):
        ax[0].plot(t, np.hypot(*res[i[0]]["cap"].T.astype(float)), col, label=lab)
ax[0].set_xlabel("ns into the pulse capture"); ax[0].set_ylabel("|IQ|"); ax[0].legend()
ax[1].hist(L, 20); ax[1].set_xlabel("loop latency at the buffer (ns)")
plt.tight_layout(); plt.show()
np.savez(OUT_DIR / "feed_forward_run.npz", rows=rows, y=y, came=came, dec=dec, gate=g, L_buf=np.array([r["L_buf"] for r in res]))

**Notes**
- The latency here is measured at the readout buffer: from the window's last sample to the pulse's 50% point, both seen
  through the loopback. The paper's headline $L_{50}$ (174.2 or 176.0 ns, depending on the bitstream load) comes from
  NN_live's own stream timestamps (`seg_stamp`) and is defined in paper Sec. IX; the two use different reference points,
  so expect them to differ by a few ns.
- **Things to try:**
  - Make `SLACK` more negative. Below some point the decision arrives after the window opens and the gate blanks every
    pulse as late; the paper's Fig. 8(b) sweeps exactly this.
  - Set `pass0=1` to play the pulse on either decision.
  - Change `CP_GAIN`.
- The tProc can also act on the decision itself: wait for it, read input port 3, branch, and play the pulse. That path
  is slower (about 337 ns) but can choose between waveforms. `analysis_scripts/diagnostics/tproc_path_v143.py` measures it.